# 15 · Feature Minimality and Invariance Test

This notebook asks a sharper version of the universality question:

> how little local information is needed to recover the zeta–GUE closeness signal?

Instead of relying on one fixed descriptor set, we test several progressively simpler representations of local windows.

## Feature families tested

For normalized 5-gap windows, we compare:

1. **full**  
   raw coordinates + summary descriptors

2. **shape-only**  
   just the normalized window coordinates

3. **summary-only**  
   unevenness, symmetry, entropy, ratio statistics

4. **ratio-only**  
   only local consecutive-ratio information

5. **entropy+symmetry**  
   just two compressed statistics

6. **rank-order only**  
   replace values by their within-window ranks

7. **random projection**  
   random low-dimensional projection of the normalized window

## Goal

For each representation, we ask:

- Is zeta closer to GUE than to Poisson?
- Does that remain true across several distance measures?

This is a direct feature-minimality / invariance test.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
# Zeta gaps
N = 1000
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

# Poisson control
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))

# Finite GUE bulk gaps
def sample_gue_bulk_spacings(matrix_size=140, n_matrices=50, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=140, n_matrices=50, bulk_fraction=0.5, rng=rng)

len(zeta_gaps), len(poisson_gaps), len(gue_gaps)

## Window and descriptor helpers

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_features(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return np.array([np.mean(r), np.std(r)], dtype=float)

def rank_order_features(window):
    order = np.argsort(np.argsort(window))
    return order.astype(float)

def descriptor_vector(window, mode="full", proj_matrix=None):
    ratio_vec = ratio_features(window)
    summary_vec = np.array([
        unevenness(window),
        reversal_symmetry_score(window),
        window_entropy(window),
        ratio_vec[0],
        ratio_vec[1],
    ], dtype=float)

    if mode == "full":
        return np.concatenate([window, summary_vec])
    elif mode == "shape_only":
        return window.copy()
    elif mode == "summary_only":
        return summary_vec.copy()
    elif mode == "ratio_only":
        return ratio_vec.copy()
    elif mode == "entropy_symmetry":
        return np.array([window_entropy(window), reversal_symmetry_score(window)], dtype=float)
    elif mode == "rank_only":
        return rank_order_features(window)
    elif mode == "random_projection":
        return window @ proj_matrix
    else:
        raise ValueError(f"Unknown mode: {mode}")

def build_descriptors(gaps, k=5, mode="full", proj_matrix=None):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    desc = np.array([descriptor_vector(w, mode=mode, proj_matrix=proj_matrix) for w in windows_n])
    return windows_n, desc

## PCA + distance helpers

In [ ]:
def pca_embed_three(A, B, C):
    all_desc = np.vstack([A, B, C])
    mean = all_desc.mean(axis=0)
    std = all_desc.std(axis=0)
    std[std == 0] = 1.0

    A_std = (A - mean) / std
    B_std = (B - mean) / std
    C_std = (C - mean) / std

    X = np.vstack([A_std, B_std, C_std])
    Xc = X - X.mean(axis=0)

    cov = np.cov(Xc, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvecs = eigvecs[:, order]
    comps = eigvecs[:, :2]

    emb = Xc @ comps
    nA, nB, nC = len(A), len(B), len(C)
    return emb[:nA], emb[nA:nA+nB], emb[nA+nB:]

def centroid_distance(A, B):
    return float(np.linalg.norm(A.mean(axis=0) - B.mean(axis=0)))

def density_grid(X, bins_x, bins_y):
    H, _, _ = np.histogram2d(X[:,0], X[:,1], bins=[bins_x, bins_y], density=True)
    return H

def grid_l2_distance(H1, H2):
    return float(np.linalg.norm(H1.ravel() - H2.ravel()))

def wasserstein_1d(a, b):
    a = np.sort(np.asarray(a))
    b = np.sort(np.asarray(b))
    n = min(len(a), len(b))
    if len(a) != n:
        idx = np.linspace(0, len(a) - 1, n).astype(int)
        a = a[idx]
    if len(b) != n:
        idx = np.linspace(0, len(b) - 1, n).astype(int)
        b = b[idx]
    return float(np.mean(np.abs(a - b)))

def compute_metrics(zeta_desc, poisson_desc, gue_desc):
    zeta_emb, poisson_emb, gue_emb = pca_embed_three(zeta_desc, poisson_desc, gue_desc)

    xmin = min(zeta_emb[:,0].min(), poisson_emb[:,0].min(), gue_emb[:,0].min())
    xmax = max(zeta_emb[:,0].max(), poisson_emb[:,0].max(), gue_emb[:,0].max())
    ymin = min(zeta_emb[:,1].min(), poisson_emb[:,1].min(), gue_emb[:,1].min())
    ymax = max(zeta_emb[:,1].max(), poisson_emb[:,1].max(), gue_emb[:,1].max())

    bins_x = np.linspace(xmin, xmax, 30)
    bins_y = np.linspace(ymin, ymax, 30)

    zeta_H = density_grid(zeta_emb, bins_x, bins_y)
    poisson_H = density_grid(poisson_emb, bins_x, bins_y)
    gue_H = density_grid(gue_emb, bins_x, bins_y)

    return {
        "centroid_zeta_GUE": centroid_distance(zeta_emb, gue_emb),
        "centroid_zeta_Poisson": centroid_distance(zeta_emb, poisson_emb),
        "centroid_gap": centroid_distance(zeta_emb, poisson_emb) - centroid_distance(zeta_emb, gue_emb),
        "grid_zeta_GUE": grid_l2_distance(zeta_H, gue_H),
        "grid_zeta_Poisson": grid_l2_distance(zeta_H, poisson_H),
        "grid_gap": grid_l2_distance(zeta_H, poisson_H) - grid_l2_distance(zeta_H, gue_H),
        "wasserstein_pc1_gap": wasserstein_1d(zeta_emb[:,0], poisson_emb[:,0]) - wasserstein_1d(zeta_emb[:,0], gue_emb[:,0]),
        "wasserstein_pc2_gap": wasserstein_1d(zeta_emb[:,1], poisson_emb[:,1]) - wasserstein_1d(zeta_emb[:,1], gue_emb[:,1]),
        "zeta_emb": zeta_emb,
        "poisson_emb": poisson_emb,
        "gue_emb": gue_emb,
    }

## Evaluate several feature families

We use:
- 5-gap windows
- first 700 gaps for zeta and Poisson
- matched prefix for GUE windows

In [ ]:
modes = [
    "full",
    "shape_only",
    "summary_only",
    "ratio_only",
    "entropy_symmetry",
    "rank_only",
    "random_projection",
]

zeta_base = zeta_gaps[:700]
poisson_base = poisson_gaps[:700]
gue_base = gue_gaps[:950]

results = []

for mode in modes:
    proj_matrix = None
    if mode == "random_projection":
        proj_rng = np.random.default_rng(9423)
        proj_matrix = proj_rng.normal(size=(5, 3))

    _, zeta_desc = build_descriptors(zeta_base, k=5, mode=mode, proj_matrix=proj_matrix)
    _, poisson_desc = build_descriptors(poisson_base, k=5, mode=mode, proj_matrix=proj_matrix)
    _, gue_desc = build_descriptors(gue_base, k=5, mode=mode, proj_matrix=proj_matrix)

    m = min(len(zeta_desc), len(poisson_desc), len(gue_desc))
    metrics = compute_metrics(zeta_desc[:m], poisson_desc[:m], gue_desc[:m])

    results.append({
        "mode": mode,
        "centroid_gap": metrics["centroid_gap"],
        "grid_gap": metrics["grid_gap"],
        "wasserstein_pc1_gap": metrics["wasserstein_pc1_gap"],
        "wasserstein_pc2_gap": metrics["wasserstein_pc2_gap"],
        "zeta_emb": metrics["zeta_emb"],
        "poisson_emb": metrics["poisson_emb"],
        "gue_emb": metrics["gue_emb"],
    })

results

## Gap summary table

Positive gaps mean:
\[
d(\text{zeta}, \text{Poisson}) > d(\text{zeta}, \text{GUE})
\]
so larger is stronger evidence of invariance.

In [ ]:
gap_summary = {
    r["mode"]: {
        "centroid_gap": r["centroid_gap"],
        "grid_gap": r["grid_gap"],
        "wasserstein_pc1_gap": r["wasserstein_pc1_gap"],
        "wasserstein_pc2_gap": r["wasserstein_pc2_gap"],
    }
    for r in results
}
gap_summary

## Plot centroid gaps by feature family

In [ ]:
labels = [r["mode"] for r in results]
vals = [r["centroid_gap"] for r in results]

plt.figure(figsize=(10.5, 4.8))
plt.bar(labels, vals)
plt.xticks(rotation=25)
plt.ylabel("centroid gap")
plt.title("Feature minimality test: centroid gap")
plt.tight_layout()
plt.show()

## Plot density-grid gaps by feature family

In [ ]:
vals = [r["grid_gap"] for r in results]

plt.figure(figsize=(10.5, 4.8))
plt.bar(labels, vals)
plt.xticks(rotation=25)
plt.ylabel("grid L2 gap")
plt.title("Feature minimality test: density-grid gap")
plt.tight_layout()
plt.show()

## Plot Wasserstein gaps by feature family

In [ ]:
vals1 = [r["wasserstein_pc1_gap"] for r in results]
vals2 = [r["wasserstein_pc2_gap"] for r in results]

x = np.arange(len(labels))
width = 0.38

plt.figure(figsize=(11, 4.8))
plt.bar(x - width/2, vals1, width, label="PC1 gap")
plt.bar(x + width/2, vals2, width, label="PC2 gap")
plt.xticks(x, labels, rotation=25)
plt.ylabel("Wasserstein gap")
plt.title("Feature minimality test: Wasserstein gaps")
plt.legend()
plt.tight_layout()
plt.show()

## Representative PCA embeddings

Show a few representative feature families.

In [ ]:
show_modes = ["full", "ratio_only", "rank_only", "random_projection"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))

for ax, mode in zip(axes, show_modes):
    rec = next(r for r in results if r["mode"] == mode)
    ax.scatter(rec["zeta_emb"][:,0], rec["zeta_emb"][:,1], s=8, alpha=0.20, label="zeta")
    ax.scatter(rec["poisson_emb"][:,0], rec["poisson_emb"][:,1], s=8, alpha=0.20, label="Poisson")
    ax.scatter(rec["gue_emb"][:,0], rec["gue_emb"][:,1], s=8, alpha=0.20, label="GUE")
    ax.set_title(mode)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

axes[0].legend()
plt.tight_layout()
plt.show()

## Final summary

In [ ]:
summary = {
    "gap_summary": gap_summary,
    "all_modes_positive_centroid_gap": bool(all(r["centroid_gap"] > 0 for r in results)),
    "all_modes_positive_grid_gap": bool(all(r["grid_gap"] > 0 for r in results)),
    "all_modes_positive_wasserstein_pc1_gap": bool(all(r["wasserstein_pc1_gap"] > 0 for r in results)),
    "all_modes_positive_wasserstein_pc2_gap": bool(all(r["wasserstein_pc2_gap"] > 0 for r in results)),
}
summary

## Notes

- If the zeta–GUE closeness signal remains across very compressed or altered feature families, that suggests the universality signal is representation-invariant rather than tied to one handcrafted descriptor set.
- The rank-only representation is especially strict: it discards actual normalized values and keeps only within-window ordering information.
- The random-projection representation is a crude test of whether generic low-dimensional mixtures of window coordinates retain the signal.

## Next directions

A natural next notebook could do one of these:

1. bootstrap these feature-family comparisons for uncertainty intervals  
2. extend the same minimality test to conditional windows (after small vs after large gaps)  
3. vary the random-projection dimension and repeat several random seeds  
4. compare feature invariance across different window lengths simultaneously